Jai Mata Di


In [ ]:
!pip install -q scikit-learn pandas numpy nltk vaderSentiment gensim matplotlib seaborn gradio joblib

import nltk
nltk.download(['punkt', 'stopwords', 'wordnet', 'vader_lexicon'])
print("Libraries Installed!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import gradio as gr
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
import re
import warnings
warnings.filterwarnings('ignore')

# Load IMDB Dataset (Supervised Learning)
#  upload IMDB Dataset.csv or (use Hugging Face)
from datasets import load_dataset
dataset = load_dataset("imdb", split="train[:20000]")  # Subset for speed
df = pd.DataFrame(dataset)
df = df.rename(columns={'text': 'review', 'label': 'sentiment'})
df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'positive'})

print(df.head())
print("\nDataset Shape:", df.shape)

In [ ]:
import nltk

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    stop_words = nltk.corpus.stopwords.words('english')
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = nltk.WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess_text)
print(" Preprocessing Done!")

In [ ]:
# Feature Extraction
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Supervised Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
vader = SentimentIntensityAnalyzer()

def get_sentiment(text):
    clean_text = preprocess_text(text)

    # Supervised Model Prediction
    vector = tfidf.transform([clean_text])
    supervised_pred = model.predict(vector)[0]
    supervised_prob = model.predict_proba(vector).max()

    # VADER Lexicon (Unsupervised component)
    vader_scores = vader.polarity_scores(text)
    vader_compound = vader_scores['compound']

    # Hybrid Logic
    if vader_compound > 0.05:
        final_sentiment = "positive"
    elif vader_compound < -0.05:
        final_sentiment = "negative"
    else:
        final_sentiment = supervised_pred

    confidence = round(supervised_prob * 100, 2)
    return final_sentiment, confidence, vader_compound

In [ ]:
def get_topics(texts, num_topics=5):
    tokenized = [doc.split() for doc in texts]
    dictionary = corpora.Dictionary(tokenized)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized]

    lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=10, random_state=42)
    topics = lda_model.print_topics()
    return topics

In [ ]:
def generate_smart_response(text, sentiment):
    responses = {
        "positive": [
            "Thank you so much for your positive feedback! We're glad you loved it.",
            "We're thrilled to hear that! Your support means everything to us.",
            "Thank you! It's customers like you who make our work worthwhile."
        ],
        "negative": [
            "We're truly sorry to hear about your experience. Can you tell us more so we can improve?",
            "We apologize for the inconvenience. Our team will look into this immediately.",
            "Thank you for bringing this to our attention. We value your feedback."
        ]
    }

    import random
    base_response = random.choice(responses.get(sentiment, responses["positive"]))

    # Add personalization using keywords
    if any(word in text.lower() for word in ['slow', 'delay', 'late', 'wait']):
        base_response += " We are working on improving our response time."

    return base_response

In [ ]:
def full_analysis(text):
    if len(text.strip()) < 10:
        return "Please enter a longer text.", "N/A", 0, "No topics"

    sentiment, confidence, vader_score = get_sentiment(text)
    response = generate_smart_response(text, sentiment)

    # Get topics (Unsupervised)
    topics = get_topics([preprocess_text(text)], num_topics=3)

    result = f"""
    <h3>Sentiment: <b>{sentiment.upper()}</b> ({confidence}% confidence)</h3>
    <p><b>VADER Score:</b> {vader_score:.3f}</p>
    <hr>
    <h3>🤖 Smart Response:</h3>
    <p>{response}</p>
    """
    return result, sentiment, confidence, str(topics[0])

# Launch Interface
interface = gr.Interface(
    fn=full_analysis,
    inputs=gr.Textbox(lines=5, placeholder="Enter review or feedback..."),
    outputs=[
        gr.HTML(),
        gr.Textbox(label="Sentiment"),
        gr.Number(label="Confidence"),
        gr.Textbox(label="Key Topics (LDA)")
    ],
    title="Sentiment Analysis & Smart Response System (ML-Based)",
    description="Supervised (TF-IDF + LR) + VADER + LDA Topic Modeling",
    examples=[
        ["I absolutely love this product! Best purchase ever."],
        ["Worst service. Delivery was late and damaged."]
    ],
    theme=gr.themes.Soft()
)

interface.launch(share=True)